# Build the Dataset A canonical node map

This notebook independently derives the sole authoritative node universe from Dataset A. It never searches for, reads, compares, converts, renames, overwrites, or deletes any legacy node-map file. It never reads Dataset B.

The workflow is sequential and fail-closed. Run each cell once in order. No unsafe resume is implemented: after a runtime interruption, reconnect, remount Drive, and restart from Stage 1. The complete Dataset A scan is repeated rather than reusing unverifiable partial identity state.

## Stage 1 - Mount Google Drive

Google Drive contains the read-only Dataset A source and the private persistent output directory. This stage performs no dataset read and publishes no artifact.

**Stop condition:** stop if Drive mounting fails.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("PASS: Google Drive mounted.")

## Stage 2 - Define and validate Dataset A and output paths

Dataset A is the only source of author identities. Raw workbooks remain read-only. Candidate artifacts, the canonical Parquet file, and the validation manifest remain under the private persistent output root.

**Stop conditions:** stop if Dataset A is unavailable, a canonical path differs from the declared contract, or an output path escapes the private output root.

In [ ]:
from pathlib import Path

DATASET_A_ROOT = Path(
    "/content/drive/MyDrive/Thesis/Dataset A/core_army_pro_fans_tweets"
)
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Thesis/TDMEC_PROJECT_OUTPUTS"
)
MANIFESTS_ROOT = OUTPUT_ROOT / "manifests"
CANONICAL_NODE_MAP = MANIFESTS_ROOT / "node_index_map.parquet"
VALIDATION_MANIFEST = (
    MANIFESTS_ROOT / "node_index_map_validation_manifest.json"
)
REPO_ROOT = Path("/content/community-evolution-modeling")

assert DATASET_A_ROOT.is_dir(), "STOP: Dataset A root is unavailable."
assert CANONICAL_NODE_MAP == Path(
    "/content/drive/MyDrive/Thesis/TDMEC_PROJECT_OUTPUTS/"
    "manifests/node_index_map.parquet"
)
assert VALIDATION_MANIFEST == Path(
    "/content/drive/MyDrive/Thesis/TDMEC_PROJECT_OUTPUTS/"
    "manifests/node_index_map_validation_manifest.json"
)
assert MANIFESTS_ROOT.is_relative_to(OUTPUT_ROOT)
assert not REPO_ROOT.is_relative_to(Path("/content/drive"))

print("Dataset A available:", DATASET_A_ROOT.is_dir())
print("Canonical filename:", CANONICAL_NODE_MAP.name)
print("Validation-manifest filename:", VALIDATION_MANIFEST.name)
print("PASS: Canonical paths are fixed and privacy-scoped.")

## Stage 3 - Clone and pin the audited repository implementation

Repository code is cloned into ephemeral `/content` storage. Only the exact ephemeral checkout may be removed on rerun; nothing under `/content/drive` is deleted or moved.

**Stop conditions:** stop on unsafe deletion scope, clone failure, commit mismatch, or a dirty checkout.

In [ ]:
import shutil
import subprocess

REPOSITORY_URL = (
    "https://github.com/faezehmzf/community-evolution-modeling.git"
)
EXPECTED_SHA = "840dd94a80322083bb498a42bbd48fe4cabd85a4"

assert len(EXPECTED_SHA) == 40, "STOP: Audited commit pin is invalid."
assert REPO_ROOT == Path("/content/community-evolution-modeling")
assert REPO_ROOT.is_relative_to(Path("/content"))
assert not REPO_ROOT.is_relative_to(Path("/content/drive"))

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ["git", "clone", REPOSITORY_URL, str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", EXPECTED_SHA],
    cwd=REPO_ROOT,
    check=True,
)
observed_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()
working_tree_status = subprocess.check_output(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    text=True,
).strip()
assert observed_sha == EXPECTED_SHA, "STOP: Repository commit mismatch."
assert working_tree_status == "", "STOP: Repository checkout is dirty."

print("Audited repository commit:", observed_sha)
print("PASS: Repository clone is pinned and clean.")

## Stage 4 - Install and validate imports in the active Colab kernel

The supported `test` extra is installed with the active kernel's `sys.executable`. Standard site-directory processing refreshes a newly created editable-install `.pth` file without manual `sys.path` editing or a runtime restart.

**Stop conditions:** stop on repository-layout mismatch, pip failure, undiscoverable packages, wrong versions, modules outside the pinned checkout, or CLI failure.

In [ ]:
import importlib
import importlib.metadata
import importlib.util
import site
import sys

assert (REPO_ROOT / "pyproject.toml").is_file()
assert (REPO_ROOT / "src").is_dir()
print("sys.executable:", sys.executable)
subprocess.run(
    [sys.executable, "-m", "pip", "--version"],
    cwd=REPO_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[test]"],
    cwd=REPO_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "show", "tdmec-discovery"],
    cwd=REPO_ROOT,
    check=True,
)

site_directories = list(site.getsitepackages())
user_site = site.getusersitepackages()
site_directories.extend([user_site] if isinstance(user_site, str) else user_site)
for directory in site_directories:
    if Path(directory).is_dir():
        site.addsitedir(directory)
importlib.invalidate_caches()

package_names = (
    "tdmec",
    "tdmec_diagnostics",
    "tdmec_discovery",
    "tdmec_pilot",
)
spec_status = {
    name: importlib.util.find_spec(name) is not None
    for name in package_names
}
assert all(spec_status.values()), f"STOP: Package discovery failed: {spec_status}"

import tdmec
import tdmec_diagnostics
import tdmec_discovery
import tdmec_pilot

packages = {
    "tdmec": tdmec,
    "tdmec_diagnostics": tdmec_diagnostics,
    "tdmec_discovery": tdmec_discovery,
    "tdmec_pilot": tdmec_pilot,
}
expected_source_root = (REPO_ROOT / "src").resolve()
for name, package in packages.items():
    package_file = Path(package.__file__).resolve()
    assert package_file.is_relative_to(expected_source_root), (
        f"STOP: {name} resolved outside the pinned checkout."
    )
assert tdmec.__version__ == "0.1.0-phase1"
assert tdmec_diagnostics.__version__ == "0.2.0-phase2"
assert importlib.metadata.version("tdmec-discovery") == "0.1.0"
subprocess.run(
    [sys.executable, "-m", "tdmec_diagnostics.cli", "--help"],
    cwd=REPO_ROOT,
    check=True,
)
print("PASS: Active-kernel installation and imports verified.")

### Stage 4A - Define fail-closed builder helpers over audited APIs

These helpers reuse the pinned repository's `iter_xlsx_rows`, `validate_required_columns`, `parse_user_blob`, `normalize_account_id`, schema constants, `sha256_file`, privacy guard, and later `load_node_map`. They add only orchestration, aggregate accounting, structural validation, and atomic publication. No unrelated workbook or identity parser is introduced.

**Stop condition:** stop on any helper-definition import failure.

In [ ]:
import json
import os
import tempfile
from dataclasses import dataclass
from typing import Iterable, Sequence

import pandas as pd
from pandas.api.types import is_bool_dtype, is_integer_dtype, is_numeric_dtype

from tdmec.hashing import sha256_file
from tdmec_diagnostics.privacy import assert_privacy_safe_mapping
from tdmec_diagnostics.schema_contracts import (
    DATASET_A_ADAPTER_ID,
    DATASET_A_DOCUMENTED_COLUMNS,
    DATASET_A_REQUIRED_COLUMNS,
    DATASET_A_SHEET_NAME,
)
from tdmec_diagnostics.workbook_io import (
    UnsupportedSchemaError,
    iter_xlsx_rows,
    validate_required_columns,
)
from tdmec_pilot.identifiers import normalize_account_id
from tdmec_pilot.user_blob import parse_user_blob

CANONICAL_NODE_MAP_COLUMNS = ("author_account_id", "node_index")
EXPECTED_DATASET_A_FILENAMES = tuple(
    f"core_army_pro_fans_tweets_part_{part:03d}.xlsx"
    for part in range(1, 13)
)

class CanonicalNodeMapError(ValueError):
    pass

class CanonicalNodeMapConflictError(CanonicalNodeMapError):
    pass

@dataclass(frozen=True)
class DatasetAAuthorScan:
    author_ids: frozenset[str]
    workbook_count: int
    total_rows_inspected: int
    valid_author_record_count: int
    missing_author_record_count: int
    malformed_author_record_count: int

@dataclass(frozen=True)
class NodeMapValidation:
    row_count: int
    columns: tuple[str, str]
    index_min: int
    index_max: int
    unique_author_count: int
    unique_index_count: int
    exact_index_set: bool
    canonical_numeric_order: bool

@dataclass(frozen=True)
class CandidateNodeMap:
    path: Path
    sha256: str
    validation: NodeMapValidation

@dataclass(frozen=True)
class PublishedNodeMap:
    path: Path
    sha256: str
    published_new_file: bool
    validation: NodeMapValidation

def discover_dataset_a_workbooks(root: Path) -> list[Path]:
    if not root.is_dir():
        raise FileNotFoundError("Dataset A root is unavailable.")
    workbooks = sorted(root.glob("*.xlsx"), key=lambda path: path.name)
    if tuple(path.name for path in workbooks) != EXPECTED_DATASET_A_FILENAMES:
        raise CanonicalNodeMapError(
            "Dataset A workbook names do not match the canonical set."
        )
    if not all(path.is_file() for path in workbooks):
        raise CanonicalNodeMapError("A Dataset A workbook is not a file.")
    return workbooks

def inspect_dataset_a_workbooks(workbooks: Sequence[Path]) -> None:
    for path in workbooks:
        sheet, header, rows = iter_xlsx_rows(
            path,
            expected_sheet=DATASET_A_SHEET_NAME,
        )
        try:
            if sheet != DATASET_A_SHEET_NAME:
                raise UnsupportedSchemaError("Dataset A worksheet mismatch.")
            validate_required_columns(
                header,
                DATASET_A_REQUIRED_COLUMNS,
                adapter_id=DATASET_A_ADAPTER_ID,
                allow_extra=True,
            )
            if tuple(header) != DATASET_A_DOCUMENTED_COLUMNS:
                raise UnsupportedSchemaError(
                    "Dataset A header is not the documented 31-column schema."
                )
            next(rows, None)
        finally:
            close = getattr(rows, "close", None)
            if close is not None:
                close()

def scan_dataset_a_authors(
    workbooks: Sequence[Path],
    *,
    expected_workbook_count: int = 12,
) -> DatasetAAuthorScan:
    ordered = sorted((Path(path) for path in workbooks), key=lambda path: path.name)
    if len(ordered) != expected_workbook_count:
        raise CanonicalNodeMapError("Unexpected Dataset A workbook count.")
    author_ids = set()
    total_rows = valid = missing = malformed = 0
    for path in ordered:
        sheet, header, rows = iter_xlsx_rows(
            path,
            expected_sheet=DATASET_A_SHEET_NAME,
        )
        if sheet != DATASET_A_SHEET_NAME:
            raise UnsupportedSchemaError("Dataset A worksheet mismatch.")
        columns = validate_required_columns(
            header,
            DATASET_A_REQUIRED_COLUMNS,
            adapter_id=DATASET_A_ADAPTER_ID,
            allow_extra=True,
        )
        if tuple(header) != DATASET_A_DOCUMENTED_COLUMNS:
            raise UnsupportedSchemaError(
                "Dataset A header is not the documented 31-column schema."
            )
        user_column = columns["user"]
        for row in rows:
            total_rows += 1
            raw_user = row[user_column] if user_column < len(row) else None
            parsed = parse_user_blob(raw_user)
            if not parsed.ok:
                if parsed.error == "missing_user":
                    missing += 1
                else:
                    malformed += 1
                continue
            normalized = normalize_account_id(parsed.account_id)
            if normalized is None or normalized != parsed.account_id:
                malformed += 1
                continue
            valid += 1
            author_ids.add(normalized)
    return DatasetAAuthorScan(
        author_ids=frozenset(author_ids),
        workbook_count=len(ordered),
        total_rows_inspected=total_rows,
        valid_author_record_count=valid,
        missing_author_record_count=missing,
        malformed_author_record_count=malformed,
    )

def assert_scan_publishable(
    scan: DatasetAAuthorScan,
    *,
    expected_unique_author_count: int = 16_736,
) -> None:
    if scan.malformed_author_record_count != 0:
        raise CanonicalNodeMapError(
            "Non-null malformed author values prohibit publication."
        )
    if len(scan.author_ids) != expected_unique_author_count:
        raise CanonicalNodeMapError("Unexpected final unique author count.")

def build_canonical_node_map_frame(
    author_ids: Iterable[str],
    *,
    expected_count: int = 16_736,
) -> pd.DataFrame:
    normalized_ids = []
    for value in author_ids:
        if not isinstance(value, str) or not value.isdigit():
            raise CanonicalNodeMapError("Author ID is not an exact digit string.")
        if normalize_account_id(value) != value:
            raise CanonicalNodeMapError("Author ID normalization failed.")
        normalized_ids.append(value)
    unique_ids = set(normalized_ids)
    if len(unique_ids) != expected_count:
        raise CanonicalNodeMapError("Incorrect final node count.")
    ordered_ids = sorted(unique_ids, key=lambda value: (int(value), value))
    frame = pd.DataFrame(
        {
            "author_account_id": pd.Series(ordered_ids, dtype="string"),
            "node_index": pd.Series(range(len(ordered_ids)), dtype="int64"),
        },
        columns=list(CANONICAL_NODE_MAP_COLUMNS),
    )
    validate_canonical_node_map_frame(frame, expected_count=expected_count)
    return frame

def validate_canonical_node_map_frame(
    frame: pd.DataFrame,
    *,
    expected_count: int = 16_736,
) -> NodeMapValidation:
    if tuple(str(column) for column in frame.columns) != CANONICAL_NODE_MAP_COLUMNS:
        raise CanonicalNodeMapError("Incorrect canonical schema or column order.")
    if len(frame) != expected_count:
        raise CanonicalNodeMapError("Incorrect canonical row count.")
    authors = frame["author_account_id"]
    indices = frame["node_index"]
    if authors.isna().any() or indices.isna().any():
        raise CanonicalNodeMapError("Canonical mapping contains null values.")
    if is_numeric_dtype(authors.dtype):
        raise CanonicalNodeMapError("Author dtype is precision-unsafe numeric data.")
    if not authors.map(lambda value: isinstance(value, str)).all():
        raise CanonicalNodeMapError("Author IDs are not stored as strings.")
    if not authors.str.fullmatch(r"\d+").all():
        raise CanonicalNodeMapError("Author ID is not a digit string.")
    if not is_integer_dtype(indices.dtype) or is_bool_dtype(indices.dtype):
        raise CanonicalNodeMapError("node_index is not an integer dtype.")
    unique_authors = int(authors.nunique(dropna=False))
    unique_indices = int(indices.nunique(dropna=False))
    if unique_authors != expected_count:
        raise CanonicalNodeMapError("Duplicate author ID detected.")
    if unique_indices != expected_count:
        raise CanonicalNodeMapError("Duplicate node index detected.")
    integer_indices = indices.astype("int64")
    expected_indices = list(range(expected_count))
    exact_index_set = set(integer_indices.tolist()) == set(expected_indices)
    if not exact_index_set:
        raise CanonicalNodeMapError("Missing or out-of-range node index detected.")
    ordered = frame.assign(node_index=integer_indices).sort_values(
        "node_index",
        kind="mergesort",
    )
    if ordered["node_index"].tolist() != expected_indices:
        raise CanonicalNodeMapError("Canonical index order is incomplete.")
    ordered_authors = ordered["author_account_id"].tolist()
    numeric_authors = sorted(
        ordered_authors,
        key=lambda value: (int(value), value),
    )
    canonical_numeric_order = ordered_authors == numeric_authors
    if not canonical_numeric_order:
        raise CanonicalNodeMapError("Author IDs are not numerically ordered.")
    return NodeMapValidation(
        row_count=len(frame),
        columns=CANONICAL_NODE_MAP_COLUMNS,
        index_min=int(integer_indices.min()),
        index_max=int(integer_indices.max()),
        unique_author_count=unique_authors,
        unique_index_count=unique_indices,
        exact_index_set=exact_index_set,
        canonical_numeric_order=canonical_numeric_order,
    )

def create_candidate_node_map(
    frame: pd.DataFrame,
    private_directory: Path,
    *,
    expected_count: int = 16_736,
) -> CandidateNodeMap:
    validate_canonical_node_map_frame(frame, expected_count=expected_count)
    private_directory.mkdir(parents=True, exist_ok=True)
    handle, name = tempfile.mkstemp(
        prefix=".node-index-map-candidate-",
        suffix=".parquet",
        dir=private_directory,
    )
    os.close(handle)
    candidate_path = Path(name)
    try:
        frame.to_parquet(candidate_path, index=False)
        persisted = pd.read_parquet(candidate_path)
        validation = validate_canonical_node_map_frame(
            persisted,
            expected_count=expected_count,
        )
        return CandidateNodeMap(
            path=candidate_path,
            sha256=sha256_file(candidate_path),
            validation=validation,
        )
    except Exception:
        candidate_path.unlink(missing_ok=True)
        raise

def publish_candidate_node_map(
    candidate: CandidateNodeMap,
    destination: Path,
    *,
    expected_count: int = 16_736,
) -> PublishedNodeMap:
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        if not candidate.path.is_file():
            raise CanonicalNodeMapError("Validated candidate is absent.")
        if sha256_file(candidate.path) != candidate.sha256:
            raise CanonicalNodeMapError("Validated candidate checksum changed.")
        candidate_frame = pd.read_parquet(candidate.path)
        candidate_validation = validate_canonical_node_map_frame(
            candidate_frame,
            expected_count=expected_count,
        )
        if destination.exists():
            existing_frame = pd.read_parquet(destination)
            existing_validation = validate_canonical_node_map_frame(
                existing_frame,
                expected_count=expected_count,
            )
            same_mapping = existing_frame.reset_index(drop=True).equals(
                candidate_frame.reset_index(drop=True)
            )
            if not same_mapping:
                raise CanonicalNodeMapConflictError(
                    "Existing canonical map differs and will not be overwritten."
                )
            existing_sha = sha256_file(destination)
            return PublishedNodeMap(
                destination,
                existing_sha,
                False,
                existing_validation,
            )
        os.replace(candidate.path, destination)
        published_sha = sha256_file(destination)
        if published_sha != candidate.sha256:
            raise CanonicalNodeMapError("Published checksum mismatch.")
        published_frame = pd.read_parquet(destination)
        published_validation = validate_canonical_node_map_frame(
            published_frame,
            expected_count=expected_count,
        )
        if not published_frame.reset_index(drop=True).equals(
            candidate_frame.reset_index(drop=True)
        ):
            raise CanonicalNodeMapError("Published mapping mismatch.")
        return PublishedNodeMap(
            destination,
            published_sha,
            True,
            published_validation,
        )
    finally:
        candidate.path.unlink(missing_ok=True)

def require_canonical_node_map(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError("Canonical Dataset A node map is absent.")
    return path

def build_validation_manifest(
    *,
    published: PublishedNodeMap,
    scan: DatasetAAuthorScan,
    audited_repository_commit: str,
    execution_timestamp: str,
) -> dict:
    validation = published.validation
    manifest = {
        "artifact_type": "dataset_a_canonical_node_index_map",
        "canonical_filename": published.path.name,
        "sha256": published.sha256,
        "row_count": validation.row_count,
        "columns": list(validation.columns),
        "index_min": validation.index_min,
        "index_max": validation.index_max,
        "unique_author_count": validation.unique_author_count,
        "unique_index_count": validation.unique_index_count,
        "exact_index_set": validation.exact_index_set,
        "canonical_numeric_order": validation.canonical_numeric_order,
        "dataset_a_workbook_count": scan.workbook_count,
        "total_rows_inspected": scan.total_rows_inspected,
        "valid_author_record_count": scan.valid_author_record_count,
        "missing_author_record_count": scan.missing_author_record_count,
        "malformed_author_record_count": scan.malformed_author_record_count,
        "audited_repository_commit": audited_repository_commit,
        "execution_timestamp": execution_timestamp,
    }
    assert_privacy_safe_mapping(manifest)
    return manifest

def write_validation_manifest_atomic(manifest: dict, destination: Path) -> None:
    assert_privacy_safe_mapping(manifest)
    destination.parent.mkdir(parents=True, exist_ok=True)
    handle, name = tempfile.mkstemp(
        prefix=".node-index-map-manifest-",
        suffix=".json",
        dir=destination.parent,
        text=True,
    )
    temporary_path = Path(name)
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as stream:
            json.dump(manifest, stream, indent=2, sort_keys=True)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary_path, destination)
    finally:
        temporary_path.unlink(missing_ok=True)

print("PASS: Fail-closed builder helpers loaded over audited repository APIs.")

### Stage 4B - Run synthetic fail-closed preflight

This preflight uses only temporary synthetic data. It checks deterministic numeric ordering, exact-string tie-breaking, duplicate authors, duplicate indices, missing interior indices, malformed authors, incorrect final count, exact schema, identical mappings with identical or different valid Parquet bytes, conflicting mappings, preservation of the accepted existing checksum, and the absent-canonical hard stop.

**Stop condition:** stop on any failed synthetic assertion before reading Dataset A.

In [ ]:
import tempfile

import openpyxl

def synthetic_frame(authors, indices):
    return pd.DataFrame(
        {
            "author_account_id": pd.Series(authors, dtype="string"),
            "node_index": pd.Series(indices, dtype="int64"),
        },
        columns=list(CANONICAL_NODE_MAP_COLUMNS),
    )

def expect_error(error_type, action):
    try:
        action()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

numeric = build_canonical_node_map_frame(["10", "2", "1"], expected_count=3)
assert numeric["author_account_id"].tolist() == ["1", "2", "10"]
tied = build_canonical_node_map_frame(["2", "1", "01"], expected_count=3)
assert tied["author_account_id"].tolist() == ["01", "1", "2"]
expect_error(
    CanonicalNodeMapError,
    lambda: validate_canonical_node_map_frame(
        synthetic_frame(["1", "1", "2"], [0, 1, 2]),
        expected_count=3,
    ),
)
expect_error(
    CanonicalNodeMapError,
    lambda: validate_canonical_node_map_frame(
        synthetic_frame(["1", "2", "3"], [0, 0, 2]),
        expected_count=3,
    ),
)
expect_error(
    CanonicalNodeMapError,
    lambda: validate_canonical_node_map_frame(
        synthetic_frame(["1", "2", "3"], [0, 2, 3]),
        expected_count=3,
    ),
)
expect_error(
    CanonicalNodeMapError,
    lambda: build_canonical_node_map_frame(["1", "2"], expected_count=3),
)
for invalid in (
    numeric[["node_index", "author_account_id"]],
    numeric.assign(extra=0),
    numeric[["author_account_id"]],
):
    expect_error(
        CanonicalNodeMapError,
        lambda invalid=invalid: validate_canonical_node_map_frame(
            invalid,
            expected_count=3,
        ),
    )

with tempfile.TemporaryDirectory() as temporary_directory:
    temporary_root = Path(temporary_directory)
    workbook_path = temporary_root / "synthetic.xlsx"
    workbook = openpyxl.Workbook()
    worksheet = workbook.active
    worksheet.title = "tweets"
    worksheet.append(list(DATASET_A_DOCUMENTED_COLUMNS))
    user_column = DATASET_A_DOCUMENTED_COLUMNS.index("user")
    for user_value in ("{'id': 10}", None, "not-a-user"):
        row = [None] * len(DATASET_A_DOCUMENTED_COLUMNS)
        row[user_column] = user_value
        worksheet.append(row)
    workbook.save(workbook_path)
    scan = scan_dataset_a_authors(
        [workbook_path],
        expected_workbook_count=1,
    )
    assert scan.valid_author_record_count == 1
    assert scan.missing_author_record_count == 1
    assert scan.malformed_author_record_count == 1
    expect_error(
        CanonicalNodeMapError,
        lambda: assert_scan_publishable(scan, expected_unique_author_count=1),
    )

    canonical_path = temporary_root / "node_index_map.parquet"
    first = publish_candidate_node_map(
        create_candidate_node_map(numeric, temporary_root, expected_count=3),
        canonical_path,
        expected_count=3,
    )
    identical_candidate = create_candidate_node_map(
        numeric,
        temporary_root,
        expected_count=3,
    )
    assert identical_candidate.sha256 == first.sha256
    second = publish_candidate_node_map(
        identical_candidate,
        canonical_path,
        expected_count=3,
    )
    assert first.published_new_file is True
    assert second.published_new_file is False
    assert first.sha256 == second.sha256

    numeric.to_parquet(canonical_path, index=False, compression="gzip")
    different_existing_bytes = canonical_path.read_bytes()
    different_existing_sha = sha256_file(canonical_path)
    assert different_existing_sha != first.sha256
    different_candidate = create_candidate_node_map(
        numeric,
        temporary_root,
        expected_count=3,
    )
    assert different_candidate.sha256 != different_existing_sha
    accepted_different_bytes = publish_candidate_node_map(
        different_candidate,
        canonical_path,
        expected_count=3,
    )
    assert accepted_different_bytes.published_new_file is False
    assert accepted_different_bytes.sha256 == different_existing_sha
    assert canonical_path.read_bytes() == different_existing_bytes

    synthetic_scan = DatasetAAuthorScan(
        author_ids=frozenset({"1", "2", "10"}),
        workbook_count=1,
        total_rows_inspected=3,
        valid_author_record_count=3,
        missing_author_record_count=0,
        malformed_author_record_count=0,
    )
    synthetic_manifest = build_validation_manifest(
        published=accepted_different_bytes,
        scan=synthetic_scan,
        audited_repository_commit="0" * 40,
        execution_timestamp="2000-01-01T00:00:00Z",
    )
    assert synthetic_manifest["sha256"] == different_existing_sha

    original_bytes = canonical_path.read_bytes()
    conflicting = build_canonical_node_map_frame(
        ["1", "2", "4"],
        expected_count=3,
    )
    expect_error(
        CanonicalNodeMapConflictError,
        lambda: publish_candidate_node_map(
            create_candidate_node_map(
                conflicting,
                temporary_root,
                expected_count=3,
            ),
            canonical_path,
            expected_count=3,
        ),
    )
    assert canonical_path.read_bytes() == original_bytes
    expect_error(
        FileNotFoundError,
        lambda: require_canonical_node_map(temporary_root / "absent.parquet"),
    )

print("PASS: Embedded synthetic canonical node-map preflight succeeded.")

## Stage 5 - Discover exactly 12 Dataset A workbooks

Discovery is nonrecursive and requires the exact documented filenames from part 001 through part 012. No workbook content or author identifier is printed.

**Stop conditions:** stop on a missing, extra, renamed, duplicated, or non-file workbook entry.

In [ ]:
DATASET_A_WORKBOOKS = discover_dataset_a_workbooks(DATASET_A_ROOT)
assert len(DATASET_A_WORKBOOKS) == 12
assert tuple(path.name for path in DATASET_A_WORKBOOKS) == (
    EXPECTED_DATASET_A_FILENAMES
)
print("Dataset A workbook count:", len(DATASET_A_WORKBOOKS))
print("PASS: Exact Dataset A workbook set discovered.")

## Stage 6 - Validate worksheet and schema contracts

Every workbook must expose worksheet `tweets` with the exact documented 31-column Dataset A header. Validation uses the repository's audited streaming workbook interface and schema constants. It prints no row content or identifiers.

**Stop conditions:** stop on an unreadable workbook, missing worksheet, missing field, unexpected field, or header-order mismatch.

In [ ]:
from tdmec_diagnostics.schema_contracts import (
    DATASET_A_DOCUMENTED_COLUMNS,
    DATASET_A_SHEET_NAME,
)

assert DATASET_A_SHEET_NAME == "tweets"
assert len(DATASET_A_DOCUMENTED_COLUMNS) == 31
inspect_dataset_a_workbooks(DATASET_A_WORKBOOKS)
print("Validated workbook count:", len(DATASET_A_WORKBOOKS))
print("Validated worksheet:", DATASET_A_SHEET_NAME)
print("Validated column count:", len(DATASET_A_DOCUMENTED_COLUMNS))
print("PASS: Dataset A workbook schemas verified.")

## Stage 7 - Stream every Dataset A row with the audited parser

Each workbook is streamed separately using `iter_xlsx_rows`, `validate_required_columns`, `parse_user_blob`, and `normalize_account_id` through the repository helper. Only the small distinct-author set is retained in memory. Missing and non-null malformed author records are counted separately.

No persistent resume is implemented because a scientifically safe resume would require verified workbook checksums, parser configuration, repository commit, and an atomic identity-state transaction. If interrupted, restart from Stage 1 and rescan all workbooks.

**Stop conditions:** stop on any parser, schema, or workbook exception. Do not reuse partial in-memory results.

In [ ]:
all_author_ids = set()
total_rows_inspected = 0
valid_author_record_count = 0
missing_author_record_count = 0
malformed_author_record_count = 0

for workbook_number, workbook_path in enumerate(DATASET_A_WORKBOOKS, start=1):
    partial_scan = scan_dataset_a_authors(
        [workbook_path],
        expected_workbook_count=1,
    )
    all_author_ids.update(partial_scan.author_ids)
    total_rows_inspected += partial_scan.total_rows_inspected
    valid_author_record_count += partial_scan.valid_author_record_count
    missing_author_record_count += partial_scan.missing_author_record_count
    malformed_author_record_count += partial_scan.malformed_author_record_count
    print(
        f"Processed workbook {workbook_number}/12; "
        f"rows inspected so far: {total_rows_inspected}"
    )

DATASET_A_SCAN = DatasetAAuthorScan(
    author_ids=frozenset(all_author_ids),
    workbook_count=len(DATASET_A_WORKBOOKS),
    total_rows_inspected=total_rows_inspected,
    valid_author_record_count=valid_author_record_count,
    missing_author_record_count=missing_author_record_count,
    malformed_author_record_count=malformed_author_record_count,
)
print("Total rows inspected:", DATASET_A_SCAN.total_rows_inspected)
print("Valid author records:", DATASET_A_SCAN.valid_author_record_count)
print("Missing author records:", DATASET_A_SCAN.missing_author_record_count)
print("Malformed author records:", DATASET_A_SCAN.malformed_author_record_count)
print("PASS: Complete Dataset A streaming scan finished.")

## Stage 8 - Validate normalized Dataset A author identities

All retained author IDs must already be exact digit strings produced by the audited parser and normalizer. Non-null malformed author records prohibit publication. The distinct author universe must contain exactly 16,736 IDs.

**Stop conditions:** stop on any malformed non-null author value, non-digit normalized ID, or final distinct-author count other than 16,736.

In [ ]:
assert_scan_publishable(
    DATASET_A_SCAN,
    expected_unique_author_count=16_736,
)
assert all(
    isinstance(author_id, str) and author_id.isdigit()
    for author_id in DATASET_A_SCAN.author_ids
)
print("Unique normalized Dataset A authors:", len(DATASET_A_SCAN.author_ids))
print("PASS: Dataset A author universe is publishable.")

## Stage 9 - Build the deterministic two-column mapping

Distinct exact author strings are sorted by exact integer value. Ascending positions become node indices 0 through 16,735. The resulting DataFrame contains only `author_account_id` and `node_index` in that order.

**Stop conditions:** stop on a malformed ID, unexpected unique count, or mapping-construction failure.

In [ ]:
CANONICAL_FRAME = build_canonical_node_map_frame(
    DATASET_A_SCAN.author_ids,
    expected_count=16_736,
)
assert CANONICAL_FRAME.columns.tolist() == list(CANONICAL_NODE_MAP_COLUMNS)
assert len(CANONICAL_FRAME) == 16_736
print("Canonical row count:", len(CANONICAL_FRAME))
print("Canonical columns:", CANONICAL_FRAME.columns.tolist())
print("PASS: Deterministic canonical mapping built.")

## Stage 10 - Validate every structural invariant in memory

Validation covers exact schema and order, precision-safe string authors, integer indices, null exclusion, uniqueness, the complete index set, and numeric author ordering. No row or identifier is printed.

**Stop conditions:** stop on any failed structural invariant.

In [ ]:
IN_MEMORY_VALIDATION = validate_canonical_node_map_frame(
    CANONICAL_FRAME,
    expected_count=16_736,
)
assert IN_MEMORY_VALIDATION.row_count == 16_736
assert IN_MEMORY_VALIDATION.index_min == 0
assert IN_MEMORY_VALIDATION.index_max == 16_735
assert IN_MEMORY_VALIDATION.unique_author_count == 16_736
assert IN_MEMORY_VALIDATION.unique_index_count == 16_736
assert IN_MEMORY_VALIDATION.exact_index_set is True
assert IN_MEMORY_VALIDATION.canonical_numeric_order is True
print("PASS: In-memory canonical mapping invariants verified.")

## Stage 11 - Create and hash a fully validated private candidate

Only after the complete scan and in-memory validation pass is a temporary Parquet candidate written under the private persistent manifests directory. The candidate is read back, revalidated, and hashed with streaming SHA-256. It is not yet the canonical file.

**Stop conditions:** stop on candidate write, read-back, schema, invariant, or checksum failure.

In [ ]:
NODE_MAP_CANDIDATE = create_candidate_node_map(
    CANONICAL_FRAME,
    MANIFESTS_ROOT,
    expected_count=16_736,
)
assert NODE_MAP_CANDIDATE.path.parent == MANIFESTS_ROOT
assert NODE_MAP_CANDIDATE.path.is_file()
assert len(NODE_MAP_CANDIDATE.sha256) == 64
print("Candidate row count:", NODE_MAP_CANDIDATE.validation.row_count)
print("Candidate SHA-256:", NODE_MAP_CANDIDATE.sha256)
print("PASS: Complete private candidate validated and hashed.")

## Stage 12 - Publish atomically or accept an identical canonical file

If no canonical file exists, the validated temporary file is atomically moved into place and its checksum is reverified. If a canonical file exists, its exact schema and structural invariants are validated and its complete mapping must equal the rebuilt candidate mapping. Equal mappings are accepted without overwrite even when valid Parquet bytes differ; the existing file's SHA-256 is preserved. A different mapping causes a hard stop. The temporary candidate is removed after publication, idempotent acceptance, or conflict.

**Stop conditions:** stop on a conflicting pre-existing canonical artifact, checksum drift, content drift, or atomic publication failure.

In [ ]:
PUBLISHED_NODE_MAP = publish_candidate_node_map(
    NODE_MAP_CANDIDATE,
    CANONICAL_NODE_MAP,
    expected_count=16_736,
)
assert PUBLISHED_NODE_MAP.path == CANONICAL_NODE_MAP
assert PUBLISHED_NODE_MAP.path.is_file()
assert PUBLISHED_NODE_MAP.sha256 == NODE_MAP_CANDIDATE.sha256
print("Published new canonical file:", PUBLISHED_NODE_MAP.published_new_file)
print("Canonical SHA-256:", PUBLISHED_NODE_MAP.sha256)
print("PASS: Canonical node map published or accepted idempotently.")

## Stage 13 - Validate the published artifact with `load_node_map`

The repository's actual `load_node_map(...) -> NodeMap` interface verifies the expected count and outer index bounds. `NodeMap` is a dataclass, not a DataFrame.

**Stop conditions:** stop on any loader exception, wrong return type, wrong count, or wrong index bounds.

In [ ]:
from tdmec_pilot.node_map import NodeMap, load_node_map

LOADED_NODE_MAP = load_node_map(
    CANONICAL_NODE_MAP,
    expected_count=16_736,
    index_min=0,
    index_max=16_735,
)
assert isinstance(LOADED_NODE_MAP, NodeMap)
assert len(LOADED_NODE_MAP) == 16_736
assert LOADED_NODE_MAP.min_index == 0
assert LOADED_NODE_MAP.max_index == 16_735
print("PASS: Repository NodeMap loader validation succeeded.")

## Stage 14 - Perform supplemental published-artifact validation

The published Parquet file is separately checked for exact column order, duplicate authors, duplicate indices, missing interior indices, the exact index set, integer dtype, string author storage, and numeric author ordering.

**Stop conditions:** stop on any supplemental invariant or published checksum mismatch.

In [ ]:
import pandas as pd

from tdmec.hashing import sha256_file

PUBLISHED_FRAME = pd.read_parquet(CANONICAL_NODE_MAP)
PUBLISHED_VALIDATION = validate_canonical_node_map_frame(
    PUBLISHED_FRAME,
    expected_count=16_736,
)
assert PUBLISHED_FRAME.columns.tolist() == [
    "author_account_id",
    "node_index",
]
assert PUBLISHED_FRAME["author_account_id"].nunique() == 16_736
assert PUBLISHED_FRAME["node_index"].nunique() == 16_736
assert set(PUBLISHED_FRAME["node_index"].astype(int)) == set(range(16_736))
assert PUBLISHED_VALIDATION.exact_index_set is True
assert PUBLISHED_VALIDATION.canonical_numeric_order is True
assert sha256_file(CANONICAL_NODE_MAP) == PUBLISHED_NODE_MAP.sha256
print("Published row count:", PUBLISHED_VALIDATION.row_count)
print("Published unique author count:", PUBLISHED_VALIDATION.unique_author_count)
print("Published unique index count:", PUBLISHED_VALIDATION.unique_index_count)
print("PASS: Supplemental published-artifact validation succeeded.")

## Stage 15 - Write the privacy-safe validation manifest atomically

The manifest contains only the approved aggregate fields, filenames, commit, timestamp, and checksum. It contains no author IDs, usernames, tweet text, email addresses, credentials, or private absolute paths.

**Stop conditions:** stop on an unexpected field, privacy failure, invalid aggregate, or atomic manifest-write failure.

In [ ]:
from datetime import datetime, timezone

from tdmec_diagnostics.privacy import assert_privacy_safe_mapping

EXECUTION_TIMESTAMP = datetime.now(timezone.utc).isoformat(
    timespec="seconds"
).replace("+00:00", "Z")
VALIDATION_RECORD = build_validation_manifest(
    published=PUBLISHED_NODE_MAP,
    scan=DATASET_A_SCAN,
    audited_repository_commit=EXPECTED_SHA,
    execution_timestamp=EXECUTION_TIMESTAMP,
)
APPROVED_MANIFEST_FIELDS = {
    "artifact_type",
    "canonical_filename",
    "sha256",
    "row_count",
    "columns",
    "index_min",
    "index_max",
    "unique_author_count",
    "unique_index_count",
    "exact_index_set",
    "canonical_numeric_order",
    "dataset_a_workbook_count",
    "total_rows_inspected",
    "valid_author_record_count",
    "missing_author_record_count",
    "malformed_author_record_count",
    "audited_repository_commit",
    "execution_timestamp",
}
assert set(VALIDATION_RECORD) == APPROVED_MANIFEST_FIELDS
assert_privacy_safe_mapping(VALIDATION_RECORD)
write_validation_manifest_atomic(VALIDATION_RECORD, VALIDATION_MANIFEST)
assert VALIDATION_MANIFEST.is_file()
print("Validation-manifest filename:", VALIDATION_MANIFEST.name)
print("PASS: Privacy-safe validation manifest written atomically.")

## Stage 16 - Final fail-closed status

This final stage rechecks the canonical checksum and manifest agreement. A PASS authorizes the canonical node map as the future node universe; it does not run Phase 2 diagnostics, build graph features, execute later phases, or implement Phase 3.

**Stop conditions:** any final checksum, manifest, count, or malformed-record disagreement produces a hard stop.

In [ ]:
import json

FINAL_MANIFEST = json.loads(VALIDATION_MANIFEST.read_text(encoding="utf-8"))
assert FINAL_MANIFEST == VALIDATION_RECORD
assert FINAL_MANIFEST["sha256"] == sha256_file(CANONICAL_NODE_MAP)
assert FINAL_MANIFEST["row_count"] == 16_736
assert FINAL_MANIFEST["index_min"] == 0
assert FINAL_MANIFEST["index_max"] == 16_735
assert FINAL_MANIFEST["unique_author_count"] == 16_736
assert FINAL_MANIFEST["unique_index_count"] == 16_736
assert FINAL_MANIFEST["exact_index_set"] is True
assert FINAL_MANIFEST["canonical_numeric_order"] is True
assert FINAL_MANIFEST["dataset_a_workbook_count"] == 12
assert FINAL_MANIFEST["malformed_author_record_count"] == 0

print("Canonical filename:", CANONICAL_NODE_MAP.name)
print("Canonical SHA-256:", FINAL_MANIFEST["sha256"])
print("Unique author count:", FINAL_MANIFEST["unique_author_count"])
print("FINAL STATUS: PASS - DATASET A CANONICAL NODE MAP VALIDATED")